# 43 — Evidence ledger lab (high-fidelity AdEx)

Python companion to Studio's session **evidence cart** idea
(`studio.evidence-cart.v1`): run AdEx, build a miniature ledger entry with
SHA-256 digests over canonical JSON, and verify round-trip hashes.

## Honesty box

| | |
|---|---|
| **Proves** | Local Python ledger entries for simulation-like payloads with stable digests; AdEx runnable high-fidelity dynamics. |
| **Does not prove** | Browser Studio cart UI, async analysis jobs, or server `studio.evidence-bundle.v1` product export. |
| **Artefacts** | In-memory / optional local JSON write in the working directory only. |
| **Models** | `AdExNeuron` (polyglot-complete). |
| **Schema** | Pedagogical mirror of cart fields — not a wire-compatible client. |


In [ ]:
from __future__ import annotations

import hashlib
import json
from datetime import datetime, timezone
from typing import Any

import matplotlib.pyplot as plt
import numpy as np

from sc_neurocore.neurons.models import AdExNeuron

SCHEMA = "studio.evidence-cart.v1"  # pedagogical alignment with FE cart
print("SC-NeuroCore — NB-43 evidence ledger lab")


## 1. Run AdEx and summarise spikes


In [ ]:
neuron = AdExNeuron()
n_steps = 5000
current = 300.0
v, spikes = neuron.simulate(n_steps, current=current)
t = np.arange(len(v)) * float(neuron.dt)
fig, ax = plt.subplots(figsize=(9, 2.8))
ax.plot(t, v, lw=0.9)
ax.set_title(f"AdExNeuron — I={current}, spikes={spikes}")
ax.set_xlabel("time")
ax.set_ylabel("v")
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()


## 2. Canonical JSON + SHA-256 (cart-style digests)


In [ ]:
def canonical_json(value: Any) -> str:
    return json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=False)


def sha256_hex(value: Any) -> str:
    raw = canonical_json(value).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()


payload = {
    "model": "AdExNeuron",
    "dt": float(neuron.dt),
    "n_steps": n_steps,
    "current": current,
    "spike_count": int(spikes),
    "v_min": float(np.min(v)),
    "v_max": float(np.max(v)),
    "v_head": [float(x) for x in v[:8]],
}
payload_sha = sha256_hex(payload)
print("payload_sha256:", payload_sha)
assert len(payload_sha) == 64 and all(c in "0123456789abcdef" for c in payload_sha)


## 3. Build a miniature ledger entry and verify


In [ ]:
entry = {
    "id": "ec_demo_adex_1",
    "kind": "simulation",
    "classification": "simulation",
    "label": "Simulation: AdExNeuron",
    "source_name": "AdExNeuron",
    "queued_at_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "payload": payload,
    "payload_sha256": payload_sha,
}
# recompute like FE verifyEvidenceCartExportRoundTrip
recomputed = sha256_hex(entry["payload"])
assert recomputed == entry["payload_sha256"], "digest mismatch"
bundle = {
    "schema_version": SCHEMA,
    "entry_count": 1,
    "entries": [
        {
            "classification": entry["classification"],
            "id": entry["id"],
            "kind": entry["kind"],
            "label": entry["label"],
            "payload_sha256": entry["payload_sha256"],
            "queued_at_utc": entry["queued_at_utc"],
            "source_name": entry["source_name"],
        }
    ],
    "exported_at_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "kind_counts": {"simulation": 1},
}
bundle["bundle_sha256"] = sha256_hex(
    {k: v for k, v in bundle.items() if k != "bundle_sha256"}
)
print(json.dumps(bundle, indent=2)[:800], "...")
print("NB-43 complete: ledger digests verified in-process.")
print("Studio UI cart: studio/frontend evidence cart modules (separate product tickets).")
